# NFPP Sodium-Ion ESS Research Pipeline

This notebook implements the complete research pipeline for the Sodium Iron Pyrophosphate (NFPP) battery system.

In [ ]:
import os
import subprocess
import sys

# Set PYTHONPATH to include current directory
os.environ['PYTHONPATH'] = os.environ.get('PYTHONPATH', '') + ':' + os.getcwd()

import pybamm
import numpy as np
import matplotlib.pyplot as plt
print("Environment initialized.")

## Stage 1: Cell Verification
Verify the base parameter set against literature-referenced material properties.

In [ ]:
# state: CALIBRATING
from nfpp_sodium_ion.src.calibration.verify_parameters import verify
from nfpp_sodium_ion.src.cell_parameters.cell_alpha import get_parameter_values

print("Stage 1: Running Cell Verification...")
verify()
base_params = get_parameter_values()

## Stage 2: Cell Optimization
Two-step optimization: Electrolyte (Cost) and Hessian-based Design Parameters.

In [ ]:
# state: FILTERING
from src.optimization_pipeline.optimize import NFPPoptimizer
print("Stage 2: Running Two-Step Optimization...")
optimizer = NFPPoptimizer()
optimization_results = optimizer.run_optimization()

print("\nOptimization Summary:")
print(f"Optimal Electrolyte (NaPF6, NaDFOB, FEC, VC): {optimization_results['electrolyte']}")
print(f"Optimal Design [L_c, L_a, eps_c, eps_a, r_p]: {optimization_results['design']}")

## Stage 3: Stability Validation
Comprehensive performance evaluation and resistance profile generation.

In [ ]:
# state: REDUCING
from src.optimization_pipeline.validate import StabilityValidator
print("Stage 3: Running Stability Validation...")
validator = StabilityValidator(base_params)
validation_results = validator.validate_electrochemical_performance(optimized_subset=optimization_results)

print(f"\nValidation Passed: {validation_results['met_constraints']}")

# Plot Resistance Profile
rp = validation_results['resistance_profile']
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(rp['temperature'], rp['resistance'])
plt.xlabel('Temperature [K]')
plt.ylabel('Equivalent Resistance [Ohm]')
plt.title('Resistance vs Thermal Field')

plt.subplot(1, 2, 2)
plt.plot(rp['strain'], rp['resistance'])
plt.xlabel('Thermoelastic Strain')
plt.title('Resistance vs Strain')
plt.tight_layout()
plt.show()

# Export for MATLAB
validator.export_to_matlab(validation_results, output_path="src/control_systems/optimized_params.mat")

## Stage 4: BMS Controller Validation
Verify stability and response characteristics of the BMS ECU.

In [ ]:
# state: CONDITIONING
print("Stage 4: Running MATLAB BMS Controller Validation...")
matlab_cmd = "matlab -nodisplay -nosplash -nodesktop -r \"run('src/control_systems/validate_controller.m'); quit;\""
matlab_status = "SKIPPED (MATLAB not found)"

try:
    subprocess.check_call(["which", "matlab"], stdout=subprocess.DEVNULL)
    subprocess.run(matlab_cmd, shell=True, check=True)
    matlab_status = "SUCCESS"
    print("MATLAB controller validation completed.")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("MATLAB not installed or execution failed.")

bms_report_html = f"""
<html>
<head><title>BMS Controller Validation</title></head>
<body>
<h1>BMS Controller Validation Report</h1>
<p>Status: COMPLETE</p>
<p>MATLAB Execution: {matlab_status}</p>
<h2>Performance Metrics</h2>
<ul>
    <li>Energy Capacity: {validation_results['energy_capacity_kwh']:.2f} kWh</li>
    <li>Nominal Voltage: {validation_results['nominal_voltage_v']:.2f} V</li>
    <li>Continuous Current: {validation_results['continuous_current_a']:.2f} A</li>
    <li>Cycle Life: {validation_results['cycle_life']} cycles</li>
</ul>
</body>
</html>
"""

with open("src/control_systems/report.html", "w") as f:
    f.write(bms_report_html)

print("BMS response report generated at src/control_systems/report.html")